# Week 12 exercises — set-up
**27200 Data-driven Bioengineering · AI in biology and agentic workflows**

This notebook gets everything ready in about five minutes. Run the cells from top to bottom.

1. Upload the exercises zip
2. Connect your free API key
3. Check that it works — wait for **Ready.**

Then you work in the Colab **Terminal** (step 4).

## 1 · Upload the exercises
Run the cell, click **Choose files**, and pick the exercises zip you downloaded from DTU Learn.

In [ ]:
from google.colab import files
import zipfile, os, shutil

uploaded = files.upload()                      # pick the exercises .zip
zip_name = next(iter(uploaded))

os.makedirs('/content/exercise', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/_unzip')

# find the folder that holds the three exercises, wherever the zip put it
for root, dirs, _ in os.walk('/content/_unzip'):
    if '1_proteomics' in dirs:
        for name in os.listdir(root):
            src, dst = os.path.join(root, name), os.path.join('/content/exercise', name)
            if os.path.isdir(dst): shutil.rmtree(dst)
            elif os.path.exists(dst): os.remove(dst)
            shutil.move(src, dst)
        break
shutil.rmtree('/content/_unzip', ignore_errors=True)
os.remove(zip_name)

print('Exercises in /content/exercise:')
for name in sorted(os.listdir('/content/exercise')):
    print('  ', name)

## 2 · Connect your API key
Before running this cell:
1. Get a free key at **aistudio.google.com/apikey** (sign in with Google → *Create API key*).
2. Click the **key icon** in the left sidebar → *Add new secret*.
   Name it `GEMINI_API_KEY`, paste the key, and switch on **Notebook access**.

The key is stored in your home folder, **not** in the exercise folders — so the agent cannot read it, and it is not included if you share your work.

In [ ]:
import os, subprocess, pathlib
from google.colab import userdata

key = userdata.get('GEMINI_API_KEY')
if not key:
    raise SystemExit('No GEMINI_API_KEY found. Add it with the key icon on the left, and switch on notebook access.')
os.environ['GEMINI_API_KEY'] = key

env_file = pathlib.Path.home() / '.gemini' / '.env'
env_file.parent.mkdir(parents=True, exist_ok=True)
env_file.write_text(f'GEMINI_API_KEY={key}\n')
env_file.chmod(0o600)

# Gemini CLI is usually pre-installed in Colab. Install it if it is not.
if subprocess.run(['which', 'gemini'], capture_output=True).returncode != 0:
    print('Installing Gemini CLI - about a minute...')
    subprocess.run(['npm', 'install', '-g', '@google/gemini-cli'], check=True, capture_output=True)

v = subprocess.run(['gemini', '--version'], capture_output=True, text=True)
print('Gemini CLI', (v.stdout or v.stderr).strip())

## 3 · Check that it works
This uses one request of your daily quota.

In [ ]:
import subprocess, os

def ask(extra):
    try:
        r = subprocess.run(['gemini', *extra, '-p', 'Reply with exactly one word: Ready.'],
                           capture_output=True, text=True, cwd='/content/exercise', env=os.environ, timeout=180)
    except subprocess.TimeoutExpired:
        return False, 'Gemini did not answer within 3 minutes.'
    return r.returncode == 0 and 'ready' in r.stdout.lower(), (r.stdout + r.stderr).strip()

ok, out = ask([])
if not ok:
    # the free key only reaches the Flash models; try one explicitly
    ok, out = ask(['-m', 'gemini-flash-latest'])
    if ok:
        with open(os.path.expanduser('~/.bashrc'), 'a') as f:
            f.write("\nalias gemini='gemini -m gemini-flash-latest'\n")
        print('Using a Flash model: your key does not reach the default model.')

print('Ready.' if ok else 'Something is not working. The last lines of the output were:\n\n' + out[-800:])

## 4 · Start an exercise in the Terminal
1. Click **Terminal** at the bottom-left of Colab.
2. Go to an exercise and start the agent:
```
cd /content/exercise/3_knowledge_cards
gemini
```
3. The first time, choose **Use Gemini API key**.
4. Open that exercise's `PROMPT.txt` (Files panel on the left → double-click), copy all of it, and paste it into Gemini.
5. Follow the exercise's `README.txt`.

Useful commands inside Gemini: `/stats` shows your quota · `/compress` shortens a long session · `Esc` stops the current step.

## No Terminal? Run the agent here instead
Only use this if the Terminal is not available. Set `EXERCISE` below and run the cell.

It runs the whole prompt **without stopping**, so it will not pause for you after step 1 — read its output carefully afterwards.

`--yolo` approves every action the agent takes automatically. That is acceptable **here only** because this Colab machine is temporary and the data is public. Never use it on your own computer, or with real data.

In [ ]:
EXERCISE = ''        # set to '1_proteomics', '2_visualization' or '3_knowledge_cards'

import os, re
if not EXERCISE:
    print("Set EXERCISE above, then run this cell again.")
else:
    folder = f'/content/exercise/{EXERCISE}'
    os.chdir(folder)
    prompt = open('PROMPT.txt').read()
    if EXERCISE == '2_visualization':
        # run part 1 only, as the README says: cut at the PART 2 section header
        prompt = re.split(r'\n=+\s*\nPART 2', prompt)[0]
    open('/tmp/prompt.txt', 'w').write(prompt)
    !gemini --yolo -p "$(cat /tmp/prompt.txt)"

## Download your work
Colab machines are deleted when you close the session. **Download anything you want to keep.**

Set `FOLDER` and run the cell: it zips the folder and downloads it. For exercise 3, the default, it first runs the knowledge-base check, so the zip includes an up-to-date `CHECK_REPORT.md`.

Single files: in the **Files** panel on the left, right-click a file → **Download**.

In [ ]:
FOLDER = '3_knowledge_cards/hypoxia_kb'   # or '1_proteomics', '2_visualization'

import os, shutil, subprocess, sys
from google.colab import files

path = os.path.join('/content/exercise', FOLDER)
check = os.path.join(path, 'tools', 'check_kb.py')
if os.path.exists(check):
    subprocess.run([sys.executable, check])
name = os.path.basename(path.rstrip('/'))
zip_path = shutil.make_archive(f'/content/{name}', 'zip', root_dir=os.path.dirname(path), base_dir=name)
print(f'Downloading {os.path.basename(zip_path)} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
files.download(zip_path)